# 02 · Data Assembly

Merges per-dataset label files into a unified manifest with consistent schema
(dataset, filename, path, subject, side, KL grade, split, label source). Applies
stratified subsampling to the model-predicted MRKR set to control its dominance
of the training pool.

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np
from pathlib import Path
print('Datasets:', list(config.DATASETS.keys()))

Mounted at /content/drive
Datasets: ['oai', 'nhanes3', 'mrkr', 'mendeley']


## Unified manifest construction

In [2]:
def load_labels(name, meta):
    csv=meta['labels']
    if not csv.exists(): print(f'  missing: {name}'); return pd.DataFrame()
    df=pd.read_csv(str(csv)); df.columns=[c.strip().lower() for c in df.columns]
    ren={}
    for c in ['filename','file','image','img','name']:
        if c in df.columns: ren[c]='filename'; break
    for c in ['kl_grade','kl','grade','klg','label','target']:
        if c in df.columns: ren[c]='kl_grade'; break
    for c in ['subject_id','subject','patient_id','patient','id','empi_anon']:
        if c in df.columns: ren[c]='subject_id'; break
    for c in ['side','laterality','knee']:
        if c in df.columns: ren[c]='side'; break
    for c in ['split','set','partition']:
        if c in df.columns: ren[c]='split'; break
    df=df.rename(columns=ren)
    if 'filename' not in df or 'kl_grade' not in df: return pd.DataFrame()
    out=pd.DataFrame()
    out['filename']=df['filename'].astype(str)
    out['kl_grade']=pd.to_numeric(df['kl_grade'],errors='coerce')
    out['subject_id']=df['subject_id'].astype(str) if 'subject_id' in df else 'unknown'
    out['side']=df['side'].astype(str) if 'side' in df else 'NA'
    out['split']=df['split'].astype(str) if 'split' in df else 'train'
    out['dataset']=name; out['kl_source']=meta['kl_source']
    out=out.dropna(subset=['kl_grade']); out=out[out['kl_grade'].between(0,4)]
    out['kl_grade']=out['kl_grade'].astype(int)
    return out

frames=[load_labels(n,m) for n,m in config.DATASETS.items()]
manifest=pd.concat(frames, ignore_index=True)
for n,fr in zip(config.DATASETS, frames): print(f'  {n:10s}: {len(fr):>7,}')
print(f'  TOTAL     : {len(manifest):>7,}')

  oai       :   8,547
  nhanes3   :   4,785
  mrkr      :  93,053
  mendeley  :   8,260
  TOTAL     : 114,645


## MRKR stratified subsampling

The model-predicted MRKR set is capped to limit its share of the training pool while preserving its KL-grade distribution.

In [3]:
if config.MRKR_CAP is not None:
    mrkr=manifest[manifest['dataset']=='mrkr']
    if len(mrkr)>config.MRKR_CAP:
        frac=config.MRKR_CAP/len(mrkr)
        kept=mrkr.groupby('kl_grade', group_keys=False).apply(
            lambda g: g.sample(max(1,int(round(len(g)*frac))), random_state=0))
        manifest=pd.concat([manifest[manifest['dataset']!='mrkr'], kept], ignore_index=True)
        print(f'MRKR capped: {len(mrkr):,} -> {len(kept):,}')
print(f'Manifest after cap: {len(manifest):,}')
print(pd.crosstab(manifest['dataset'], manifest['kl_grade'], margins=True))
manifest.to_csv(str(config.MANIFEST_CSV), index=False)
print(f'Saved: {config.MANIFEST_CSV}')

MRKR capped: 93,053 -> 39,999
Manifest after cap: 61,591
kl_grade      0     1      2     3     4    All
dataset                                        
mendeley   3253  1495   2175  1086   251   8260
mrkr      13671  2675  13893  6041  3719  39999
nhanes3    2563   546   1271   307    98   4785
oai        3326  1523   2241  1178   279   8547
All       22813  6239  19580  8612  4347  61591


/tmp/ipykernel_7230/3619066361.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  kept=mrkr.groupby('kl_grade', group_keys=False).apply(


Saved: /content/drive/MyDrive/Master Thesis/scope3/manifest.csv
